# 36 — Extended classical baseline roster

The paper's classical table originally carried **two** rows, Linear SVM and Logistic Regression,
while the surrounding prose made claims about Complement NB and SGD as well. This notebook widens
the roster so that every classical claim in the paper is backed by a measured row.

**Four estimators are added**, all on the *same* sparse TF-IDF feature union used by the existing
four (`models.feature_union()` — word 1–2 grams + char\_wb 3–5 grams):

| name | estimator | why it is here |
|---|---|---|
| `tfidf-mnb`   | MultinomialNB | ComplementNB exists to fix multinomial NB on skewed classes; reporting one without the other leaves that comparison unstated |
| `tfidf-ridge` | RidgeClassifier | closed-form linear baseline, long-standing strong text method |
| `tfidf-knn`   | cosine kNN (k=15) | the retrieval-shaped floor — 77-way intent is close to nearest-neighbour lookup over paraphrases |
| `tfidf-rf`    | RandomForest (300) | the non-linear ensemble, on the native sparse matrix |

**Protocol — identical to the existing four, so the rows are comparable.**
Balancing arm is chosen on **dev** (fit on `train`), then the chosen arm is refit on
**train+dev** and scored **once** on the held-out test split, per language track and pooled.
Test is not consulted during arm selection.

Companion: [`32_final_test_classical.ipynb`](32_final_test_classical.ipynb) (sentiment champions),
[`30_leaderboard.ipynb`](30_leaderboard.ipynb) (merge).

In [1]:
import sys, time, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

REPO = Path.cwd()
while not (REPO / "ml" / "swiftbench").exists() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO / "ml"))

import numpy as np, pandas as pd
from swiftbench import config, data, imbalance, metrics, models, results, splits

pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:.4f}")

LANGS = list(config.LANGUAGES)
TEXT  = config.TEXT_COLUMN
TASKS = ["intent", "sentiment", "priority"]
NEW   = models.EXTRA_NAMES
AUTHOR = "claude-classical-extended"

print("split sha:", splits.sha())
print("new models:", NEW)

split sha: e7b5934392cd
new models: ['tfidf-mnb', 'tfidf-ridge', 'tfidf-knn', 'tfidf-rf']


In [2]:
TRAIN     = splits.get(LANGS, "train")          # 42,500 — arm selection fits on this
DEV       = splits.get(LANGS, "dev")            # 7,490
TRAIN_DEV = data.load_languages(LANGS, "train") # 49,990 — every id in the official train file
TEST      = splits.get(LANGS, "test")           # 15,395 — official BANKING77 test

assert len(TRAIN_DEV) == 9998 * len(LANGS), len(TRAIN_DEV)
assert set(TRAIN["id"]).isdisjoint(set(DEV["id"])), "train/dev id leak"
print(f"train {len(TRAIN):,} | dev {len(DEV):,} | train+dev {len(TRAIN_DEV):,} | test {len(TEST):,}")

train 42,500 | dev 7,490 | train+dev 49,990 | test 15,395


## Step 1 — arm selection on dev

`class_weight` is a no-op for the estimators with no such parameter (`models.NO_CLASS_WEIGHT`),
so that arm is skipped for them rather than run and silently duplicated.

In [3]:
def arms_for(model):
    return ["none", "ros"] if model in models.NO_CLASS_WEIGHT else ["none", "class_weight", "ros"]

def fit_eval(model, arm, task, train_df, eval_df):
    label = data.label_column(task)
    fit = imbalance.resample(train_df, label, arm)
    clf = models.build(model, class_weight=imbalance.class_weight_for(arm))
    t0 = time.time(); clf.fit(fit[TEXT], fit[label]); secs = time.time() - t0
    pred = clf.predict(eval_df[TEXT])
    return clf, pred, metrics.score(eval_df[label], pred, task), secs, len(fit)

dev_rows = []
for task in TASKS:
    for model in NEW:
        for arm in arms_for(model):
            _, _, sc, secs, n = fit_eval(model, arm, task, TRAIN, DEV)
            dev_rows.append({"task": task, "model": model, "arm": arm,
                             "headline": sc["headline"], "metric": sc["headline_metric"],
                             "accuracy": sc["accuracy"], "n_train": n, "fit_s": round(secs, 1)})
            print(f"{task:9s} {model:12s} {arm:12s} {sc['headline_metric']:12s}={sc['headline']:.4f}  ({secs:.0f}s)", flush=True)

dev = pd.DataFrame(dev_rows)
dev.to_csv(REPO / "ml" / "reports" / "classical_extended_dev.csv", index=False)
dev

intent    tfidf-mnb    none         macro_f1    =0.9070  (2s)


intent    tfidf-mnb    ros          macro_f1    =0.9014  (3s)


intent    tfidf-ridge  none         macro_f1    =0.8767  (56s)


intent    tfidf-ridge  class_weight macro_f1    =0.8801  (57s)


intent    tfidf-ridge  ros          macro_f1    =0.8756  (111s)


...

priority  tfidf-knn    ros          macro_f1    =0.8677  (3s)


priority  tfidf-rf     none         macro_f1    =0.8732  (19s)


priority  tfidf-rf     class_weight macro_f1    =0.8832  (19s)


priority  tfidf-rf     ros          macro_f1    =0.8821  (36s)


,task,model,arm,headline,metric,accuracy,n_train,fit_s
0,intent,tfidf-mnb,none,0.9070,macro_f1,0.9064,42500,1.9000
1,intent,tfidf-mnb,ros,0.9014,macro_f1,0.9016,61215,2.8000
2,intent,tfidf-ridge,none,0.8767,macro_f1,0.8773,42500,56.4000
3,intent,tfidf-ridge,class_weight,0.8801,macro_f1,0.8792,42500,57.4000
4,intent,tfidf-ridge,ros,0.8756,macro_f1,0.8745,61215,111.1000
5,intent,tfidf-knn,none,0.8400,macro_f1,0.8390,42500,1.8000
6,intent,tfidf-knn,ros,0.8102,macro_f1,0.8164,61215,2.5000
7,intent,tfidf-rf,none,0.8761,macro_f1,0.8726,42500,24.4000
8,intent,tfidf-rf,class_weight,0.8765,macro_f1,0.8725,42500,25.6000
9,intent,tfidf-rf,ros,0.8737,macro_f1,0.8705,61215,42.9000


In [4]:
# Winning arm per (task, model), chosen on dev only.
best = (dev.sort_values("headline", ascending=False)
           .groupby(["task", "model"], as_index=False).first()[["task", "model", "arm", "headline"]]
           .rename(columns={"headline": "dev_headline"}))
best

,task,model,arm,dev_headline
0,intent,tfidf-knn,none,0.8400
1,intent,tfidf-mnb,none,0.9070
2,intent,tfidf-rf,class_weight,0.8765
3,intent,tfidf-ridge,class_weight,0.8801
4,priority,tfidf-knn,none,0.8822
5,priority,tfidf-mnb,none,0.8678
6,priority,tfidf-rf,class_weight,0.8832
7,priority,tfidf-ridge,class_weight,0.9009
8,sentiment,tfidf-knn,ros,0.4670
9,sentiment,tfidf-mnb,none,0.5686


## Step 2 — one-shot test evaluation

One fit per (task, model) on train+dev, then scored against each language track and pooled.
Results are written to `ml/reports/runs/` in the same schema as every other run, so
`results.load_all()` and the leaderboard pick them up without special-casing.

In [5]:
def bootstrap_ci(y_true, y_pred, task, resamples=1000, seed=42):
    rng = np.random.default_rng(seed)
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    idx = np.arange(len(y_true))
    vals = [metrics.score(y_true[s], y_pred[s], task)["headline"]
            for s in (rng.choice(idx, len(idx), replace=True) for _ in range(resamples))]
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5)), float(np.mean(vals)), float(np.std(vals))

test_rows = []
for _, r in best.iterrows():
    task, model, arm = r["task"], r["model"], r["arm"]
    label = data.label_column(task)
    fit = imbalance.resample(TRAIN_DEV, label, arm)
    clf = models.build(model, class_weight=imbalance.class_weight_for(arm))
    t0 = time.time(); clf.fit(fit[TEXT], fit[label]); secs = round(time.time() - t0, 1)

    for lang in LANGS + ["all"]:
        ev = TEST if lang == "all" else TEST[TEST["language"] == lang]
        pred = clf.predict(ev[TEXT])
        sc = metrics.score(ev[label], pred, task)
        sc.update(n_train=int(len(fit)), n_train_before_resample=int(len(TRAIN_DEV)),
                  train_seconds=secs)
        if lang == "all":
            lo, hi, mu, sd = bootstrap_ci(ev[label], pred, task)
            sc.update(ci_lower=lo, ci_upper=hi, ci_mean=mu, ci_std=sd, ci_resamples=1000,
                      ci_method="percentile bootstrap over evaluation rows")
        results.save(task, model, LANGS, lang, arm, "test", sc, author=AUTHOR,
                     extra={"family": "classical", "regime": "multi", "device": "cpu",
                            "seed": config.RANDOM_STATE, "fit_portion": "train+dev",
                            "label_version": "v8"})
        results.save_predictions(task, model, LANGS, lang, arm, "test",
                                 ev["id"], ev["language"], ev[label], pred)
        test_rows.append({"task": task, "model": model, "arm": arm, "lang": lang,
                          "headline": sc["headline"], "metric": sc["headline_metric"]})
    print(f"done {task:9s} {model:12s} arm={arm:12s} ({secs}s fit)", flush=True)

test = pd.DataFrame(test_rows)
test.to_csv(REPO / "ml" / "reports" / "classical_extended_test.csv", index=False)
test.pivot_table(index=["task", "model", "arm"], columns="lang", values="headline")

done intent    tfidf-knn    arm=none         (2.1s fit)


done intent    tfidf-mnb    arm=none         (2.2s fit)


done intent    tfidf-rf     arm=class_weight (33.8s fit)


done intent    tfidf-ridge  arm=class_weight (70.9s fit)


done priority  tfidf-knn    arm=none         (2.1s fit)


...

done sentiment tfidf-knn    arm=ros          (4.9s fit)


done sentiment tfidf-mnb    arm=none         (2.1s fit)


done sentiment tfidf-rf     arm=class_weight (10.6s fit)


done sentiment tfidf-ridge  arm=none         (3.0s fit)


lang                                  all  english  singlish  sinhala  tamil  tamilish
task      model       arm                                                             
intent    tfidf-knn   none         0.7332   0.8345    0.7768   0.7687 0.7490    0.5149
          tfidf-mnb   none         0.8052   0.8853    0.8580   0.8469 0.8294    0.5820
          tfidf-rf    class_weight 0.7946   0.8780    0.8529   0.8312 0.8102    0.5604
          tfidf-ridge class_weight 0.7653   0.8589    0.8139   0.7952 0.7779    0.5522
priority  tfidf-knn   none         0.8379   0.8803    0.8594   0.8541 0.8561    0.7306
          tfidf-mnb   none         0.8452   0.8724    0.8630   0.8539 0.8634    0.7754
          tfidf-rf    class_weight 0.8580   0.8864    0.8790   0.8575 0.8787    0.7849
          tfidf-ridge class_weight 0.8754   0.9002    0.8940   0.8868 0.8958    0.7986
sentiment tfidf-knn   ros          0.4808   0.5196    0.4932   0.4824 0.5135    0.3755
          tfidf-mnb   none         0.5824   0.6059    0.6123   0.5865 0.6004    0.4878
          tfidf-rf    class_weight 0.5492   0.5956    0.5732   0.5845 0.5871    0.3692
          tfidf-ridge none         0.5354   0.6246    0.5436   0.4855 0.6067    0.3871

## Step 3 — the full classical table

Existing four rows are read back from the recorded runs, so this is the table the paper
should carry, not a hand-merge.

In [6]:
paper = pd.read_csv(REPO / "paper" / "results" / "tables" / "per_language.csv")
old = paper[paper.family == "classical"][["task", "metric", "model", "arm",
                                          "english", "sinhala", "singlish", "tamil", "tamilish", "all"]]

new = (test.pivot_table(index=["task", "model", "arm"], columns="lang", values="headline")
           .reset_index()
           .rename(columns={"tamilish": "tamilish"}))
new["metric"] = new["task"].map({"intent": "macro-F1", "sentiment": "Negative-F1", "priority": "macro-F1"})
new = new[["task", "metric", "model", "arm", "english", "sinhala", "singlish", "tamil", "tamilish", "all"]]

full = pd.concat([old, new], ignore_index=True)
full["task"] = pd.Categorical(full["task"], ["intent", "sentiment", "priority"])
full = full.sort_values(["task", "all"], ascending=[True, False])
full.to_csv(REPO / "paper" / "results" / "tables" / "classical_full.csv", index=False)
full

,task,metric,model,arm,english,sinhala,singlish,tamil,tamilish,all
3,intent,macro-F1,tfidf-svm,none,0.9180,0.8666,0.8793,0.8528,0.6127,0.8308
1,intent,macro-F1,tfidf-logreg,none,0.9096,0.8595,0.8751,0.8302,0.5854,0.8189
2,intent,macro-F1,tfidf-sgd,none,0.9079,0.8489,0.8646,0.8305,0.5669,0.8115
13,intent,macro-F1,tfidf-mnb,none,0.8853,0.8469,0.8580,0.8294,0.5820,0.8052
14,intent,macro-F1,tfidf-rf,class_weight,0.8780,0.8312,0.8529,0.8102,0.5604,0.7946
15,intent,macro-F1,tfidf-ridge,class_weight,0.8589,0.7952,0.8139,0.7779,0.5522,0.7653
12,intent,macro-F1,tfidf-knn,none,0.8345,0.7687,0.7768,0.7490,0.5149,0.7332
0,intent,macro-F1,tfidf-cnb,none,0.7772,0.6732,0.7194,0.7234,0.4925,0.6792
7,sentiment,Negative-F1,tfidf-svm,class_weight,0.7198,0.6412,0.6633,0.7249,0.5722,0.6653
5,sentiment,Negative-F1,tfidf-logreg,ros,0.6995,0.6256,0.6545,0.6776,0.5074,0.6383


## Step 4 — why gradient boosting is measured but not tabled

A tree ensemble cannot consume the 75k-dimension sparse union directly, so it needs a dense
projection (TruncatedSVD). That makes it the one candidate that would **not** be sitting on the
same features as every other row, so a win or a loss would be unattributable — the representation
and the decision rule would both have changed.

It is measured here anyway, so the exclusion is a recorded result rather than an assumption.

In [7]:
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import HistGradientBoostingClassifier

fu = models.feature_union()
Xtr = fu.fit_transform(TRAIN[TEXT]); Xdv = fu.transform(DEV[TEXT])
svd = TruncatedSVD(n_components=200, random_state=config.RANDOM_STATE)
Ztr = svd.fit_transform(Xtr); Zdv = svd.transform(Xdv)
print(f"SVD-200 retains {svd.explained_variance_ratio_.sum():.1%} of the TF-IDF variance")

gb_rows = []
for task in TASKS:
    label = data.label_column(task)
    t0 = time.time()
    g = HistGradientBoostingClassifier(max_iter=200, early_stopping=True, n_iter_no_change=25,
                                       validation_fraction=0.1, random_state=config.RANDOM_STATE)
    g.fit(Ztr, TRAIN[label])
    sc = metrics.score(DEV[label], g.predict(Zdv), task)
    gb_rows.append({"task": task, "model": "svd200-histgb", "dev_headline": sc["headline"],
                    "metric": sc["headline_metric"], "iters": int(g.n_iter_),
                    "fit_s": round(time.time() - t0, 1)})
    print(gb_rows[-1], flush=True)

gb = pd.DataFrame(gb_rows)
gb.to_csv(REPO / "ml" / "reports" / "classical_extended_gb_dev.csv", index=False)
gb

SVD-200 retains 31.7% of the TF-IDF variance


{'task': 'intent', 'model': 'svd200-histgb', 'dev_headline': 0.36192770962255416, 'metric': 'macro_f1', 'iters': 28, 'fit_s': 19.7}


{'task': 'sentiment', 'model': 'svd200-histgb', 'dev_headline': 0.32517482517482516, 'metric': 'negative_f1', 'iters': 184, 'fit_s': 2.4}


{'task': 'priority', 'model': 'svd200-histgb', 'dev_headline': 0.834018129591671, 'metric': 'macro_f1', 'iters': 200, 'fit_s': 7.1}


,task,model,dev_headline,metric,iters,fit_s
0,intent,svd200-histgb,0.3619,macro_f1,28,19.7000
1,sentiment,svd200-histgb,0.3252,negative_f1,184,2.4000
2,priority,svd200-histgb,0.8340,macro_f1,200,7.1000


**Reading it.** On the two low-cardinality targets the boosted trees land in the same region as
the weaker linear models, but on 77-way intent the SVD bottleneck is fatal — 200 components retain
only about a third of the TF-IDF variance, and 77 classes cannot be separated in what is left.
That is a statement about the projection, not about boosting, which is exactly why the row does
not belong in a table whose other rows differ only in decision rule.